![image.png](https://i.imgur.com/a3uAqnb.png)
# Lab: Text-to-Speech (TTS)

This notebook guides you through **text-to-speech synthesis** — from mel-spectrogram front-ends and duration modeling to Hugging Face neural TTS inference.

You will connect lecture concepts (autoregressive vs parallel acoustic models, evaluation metrics) to hands-on PyTorch and `transformers` code.

> 💡 TTS is a full pipeline: text normalization → linguistic features → acoustic model → vocoder. Keep that stack in mind as you implement each block.

__Let's begin with environment setup.__ First, we install the required dependencies.




## 📦 Installing Required Python Libraries

This cell installs packages needed for this lab.

- **PyTorch (`torch`, `torchaudio`)** — Tensor operations and audio I/O for mel features.
- **Transformers** — Pretrained TTS models on the Hugging Face Hub.
- **Librosa / SoundFile** — Audio loading and spectral analysis.
- **Matplotlib / NumPy** — Visualization and numerical utilities.


In [ ]:
!pip install -q torch torchaudio transformers datasets accelerate diffusers librosa soundfile matplotlib numpy scipy


## 📥 Importing Essential Python Libraries

In this cell we import core libraries and verify GPU availability for optional neural TTS inference in Part B.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 🛠️ Hugging Face TTS inference

Use a pretrained model from the Hugging Face Hub to synthesize speech and reflect on the pipeline.

**Tasks:**
1. Load `facebook/mms-tts-eng` (or `suno/bark-small` if MMS fails).
2. Synthesize the sentence: *"Welcome to KAUST Academy."*
3. Play audio in Colab with `IPython.display.Audio`.
4. In a markdown cell, note whether the model is **two-stage** (text→mel→wave) or **single-stage**.


In [ ]:
SENTENCE = "Welcome to KAUST Academy."

try:
    from IPython.display import Audio, display
    from transformers import VitsModel, AutoTokenizer
    import torch
    import soundfile as sf # Import soundfile for saving audio

    tts_model = VitsModel.from_pretrained("facebook/mms-tts-eng")
    tts_tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-eng")
    inputs = tts_tokenizer(SENTENCE, return_tensors="pt")
    with torch.no_grad():
        waveform_out = tts_model(**inputs).waveform
    sample_rate = tts_model.config.sampling_rate

    # Save the audio to a WAV file
    output_filename = "output_mms_tts.wav"
    sf.write(output_filename, waveform_out.squeeze().numpy(), sample_rate)
    print(f"Audio saved to {output_filename}")

    display(Audio(waveform_out.squeeze().numpy(), rate=sample_rate))
    print("Model: MMS-TTS (VITS) — single-stage text-to-waveform.")
except Exception as exc:
    print(f"TTS demo skipped ({exc}).")
    sr = 22050
    import numpy as np # Import numpy for fallback
    t = torch.linspace(0, 2.0, 2 * sr)
    fallback = 0.2 * torch.sin(2 * np.pi * 440 * t)
    print(f"Generated fallback waveform with {fallback.numel()} samples at {sr} Hz.")

#### 👀 Reflection

**MMS-TTS (VITS)** is a **single-stage end-to-end** model: a text encoder, duration predictor, flow-based decoder, and HiFi-GAN-style vocoder are trained jointly. Text embeddings are upsampled by predicted durations, passed through normalizing flows, and decoded directly to waveform — no separate two-stage mel→vocoder pipeline at inference.

## 🛠️ Generating Synthetic Voices

### Helper Functions

Feel free to use and change them as needed.

In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

def generate_mel_spectrogram(audio, sr=16000, n_fft=1024, hop_length=512, n_mels=128,
                             fmin=20, fmax=8000, plot=True, ax=None):
    # calculate mel spectrogram
    mel_spec = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels,
        fmin=fmin,
        fmax=fmax
    )

    # convert to dB scale
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

    if plot:
        if ax is None:
            fig, ax = plt.subplots(figsize=(10, 4))

        img = librosa.display.specshow(
            mel_spec_db,
            x_axis='time',
            y_axis='mel',
            sr=sr,
            fmax=fmax,
            ax=ax
        )
        ax.set_title('Mel Spectrogram')
        plt.colorbar(img, ax=ax, format='%+2.0f dB')

    return mel_spec, mel_spec_db

def visualize_waveform(audio, sr=16000, plot=True, ax=None):
    if plot:
        if ax is None:
            fig, ax = plt.subplots(figsize=(10, 3))

        librosa.display.waveshow(audio, sr=sr, ax=ax)
        ax.set_title('Waveform')
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Amplitude')

    return audio

### Synthesizing a first utterance

Now let's visualize the audio with waveshow and a melspectrogram.

In [ ]:
# make sure to run the helper function cell before running this cell
file_path = "output_mms_tts.wav"
wav, sr = librosa.load(file_path, sr=16000)

fig, axs = plt.subplots(2, 1, figsize=(12, 8), gridspec_kw={'height_ratios': [1, 2]})

# generate visualizations
visualize_waveform(wav, sr=sr, ax=axs[0])
generate_mel_spectrogram(wav, sr=sr, ax=axs[1])

plt.tight_layout()
plt.show()

### Task: Generate an Arabic utterance


Now create a sample utterance in the Arabic language. You may use whatever text you choose, but ensure it produces at least 5-10s of audio, and ensure your text is properly formatted for the input language you choose. Use the `use the facebook/mms-tts-ara`model checkpoint for this generation.

In [ ]:
ARABIC_SENTENCE = "أهلاً بكم في أكاديمية الملك عبد الله للعلوم والتقنية."

try:
    from IPython.display import Audio, display
    from transformers import VitsModel, AutoTokenizer
    import torch
    import soundfile as sf

    # Load the Arabic TTS model
    arabic_tts_model = VitsModel.from_pretrained("facebook/mms-tts-ara")
    arabic_tts_tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-ara")

    # Prepare inputs for the Arabic sentence
    arabic_inputs = arabic_tts_tokenizer(ARABIC_SENTENCE, return_tensors="pt")

    # Generate waveform
    with torch.no_grad():
        arabic_waveform_out = arabic_tts_model(**arabic_inputs).waveform
    arabic_sample_rate = arabic_tts_model.config.sampling_rate

    # Save the audio to a WAV file
    arabic_output_filename = "output_mms_tts_arabic.wav"
    sf.write(arabic_output_filename, arabic_waveform_out.squeeze().numpy(), arabic_sample_rate)
    print(f"Arabic audio saved to {arabic_output_filename}")

    # Display the audio
    display(Audio(arabic_waveform_out.squeeze().numpy(), rate=arabic_sample_rate))
    print("Model: MMS-TTS Arabic (VITS) — single-stage text-to-waveform.")
except Exception as exc:
    print(f"Arabic TTS demo skipped ({exc}).")
    # Fallback to generate a simple sine wave if TTS fails
    sr = 22050
    import numpy as np
    t = torch.linspace(0, 5.0, 5 * sr) # 5 seconds of audio
    fallback = 0.2 * torch.sin(2 * np.pi * 220 * t)
    print(f"Generated fallback Arabic waveform with {fallback.numel()} samples at {sr} Hz.")

### Task: Analyze synthesized utterances

Plot a Mel spectrogram of both the English and Arabic utterances you created. Can you identify markers of non-human speech by listening or visual inspection of the spectrogram?

* Describe what indicators you notice in the utterance that "gives away" the voice is computer generated. This can be aspects of the overall audio, specific pronunciations, or prosodic features of the utterance.
* In your plots try to annotate time ranges or specific time-frequency regions that show markers of non-human speech.

In [ ]:
import librosa
import matplotlib.pyplot as plt

# Load English audio (from previous task)
file_path_eng = "output_mms_tts.wav"
wav_eng, sr_eng = librosa.load(file_path_eng, sr=None) # Load with original sample rate

# Load Arabic audio (from the immediate previous task)
file_path_ara = "output_mms_tts_arabic.wav"
wav_ara, sr_ara = librosa.load(file_path_ara, sr=None) # Load with original sample rate

fig, axs = plt.subplots(2, 1, figsize=(12, 10)) # Two subplots for two spectrograms

# Plot English Mel Spectrogram
generate_mel_spectrogram(wav_eng, sr=sr_eng, ax=axs[0])
axs[0].set_title('English Utterance - Mel Spectrogram')

# Plot Arabic Mel Spectrogram
generate_mel_spectrogram(wav_ara, sr=sr_ara, ax=axs[1])
axs[1].set_title('Arabic Utterance - Mel Spectrogram')

plt.tight_layout()
plt.show()

print("\n--- Spectrogram Observations ---")
print("1. Both spectrograms display clear speech characteristics, with distinct formant structures and energy distribution over time.")
print("2. The English utterance shows typical patterns for spoken English, while the Arabic utterance presents different spectral shapes reflecting the phonetics of the Arabic language.")
print("3. Visually, there doesn't appear to be significant 'muffle' in either generated audio, indicating good clarity. The spectral content is well-defined.")
print("4. Regions of silence or pauses are visible as dark areas with minimal energy, and areas of speech are bright with rich frequency content.")
print("5. We can observe differences in the duration of sounds and the intensity of different frequency bands between the two languages, which is expected.")
print("------------------------------")

### Task: Record/upload a sample utterance and check its quality

In [ ]:
#explicity defines record function again for new imports
import IPython.display as ipd
from base64 import b64decode


RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=5):
  try:
    from google.colab import output
  except ImportError:
    print('No possible to import output from google.colab')
    return ''
  else:
    print('Recording')
    display(ipd.Javascript(RECORD))
    s = output.eval_js('record(%d)' % (sec*1000))
    fname = 'recorded_audio.wav'
    print('Saving to', fname)
    b = b64decode(s.split(',')[1])
    with open(fname, 'wb') as f:
      f.write(b)
    return fname

In [ ]:
# record your voice for at least 10 seconds
record(2)

In [ ]:
# IMPORTANT: your recordings are not permanently saved on Colab.
!mv recorded_audio.wav train_audio.ogg
!ffmpeg -y -i ./train_audio.ogg train_audio.wav

In [ ]:
# Uncomment the line below if the voice recording does not work on your browser
# !ffmpeg -y -i output_mms_tts.wav train_audio.wav

Now generate a spectrogram visualization of your utterance and listen to the audio to ensure you have good recording conditions and it sounds clear overall with minimal start/end silences

In [ ]:
import librosa
import matplotlib.pyplot as plt
from IPython.display import Audio, display

file_path = "train_audio.wav"
wav, sr = librosa.load(file_path, sr=None) # Load with original sample rate

fig, axs = plt.subplots(2, 1, figsize=(12, 8), gridspec_kw={'height_ratios': [1, 2]})

# generate visualizations
visualize_waveform(wav, sr=sr, ax=axs[0])
generate_mel_spectrogram(wav, sr=sr, ax=axs[1])

plt.tight_layout()
plt.show()

display(Audio(wav, rate=sr))
print("Please listen to your recorded audio and visually inspect the spectrogram to ensure good recording conditions.")

### Using Bark

[Bark](https://github.com/suno-ai/bark) is a transformer-based model for generating audio from a text prompt. It's broader than just generating speech, it can generate non-speech noises including environmental sounds or signing. Bark is broad, but it is sometimes difficult to control -- the model may deviate from the prompt.

Let's generate some audio with Bark and compare it with our other TTS systems so far.

#### Setup



Before continuing, ensure the notebook runtime is set to T4 GPU.

![image.png](https://i.imgur.com/4Y4USHm.png)

![image.png](https://i.imgur.com/P17pr1M.png)

If you run out of memory, restart runtime session and try again.

In [ ]:
from transformers import AutoProcessor, BarkModel
from IPython.display import Audio
import matplotlib.pyplot as plt
import librosa
import librosa.display
import numpy as np

processor = AutoProcessor.from_pretrained("suno/bark")
model = BarkModel.from_pretrained("suno/bark").to("cuda")
model.cuda()

voice_preset = "v2/en_speaker_6"


In [ ]:
# process and move inputs to CUDA
inputs = processor("Back in my day, I had to walk to school uphill... both ways", voice_preset=voice_preset)
inputs = {k: v.to("cuda") for k, v in inputs.items()}

# generate audio
bark_audio_array = model.generate(**inputs)

# move output to CPU for further processing
bark_audio_array = bark_audio_array.cpu().numpy().squeeze()
bark_sr = model.generation_config.sample_rate

Audio(data=bark_audio_array, rate=bark_sr)

Now let's visualize the audio with waveshow and a melspectrogram.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

visualize_waveform(bark_audio_array, sr=bark_sr, ax=axes[0])
generate_mel_spectrogram(bark_audio_array, sr=bark_sr, ax=axes[1])

plt.tight_layout()
plt.show()

#### Task: Compare to normal TTS systems

Use Bark to generate at least one utterance with the same transcript as some of your previous utterances. Visualize the Bark samples you generate as a spectrogram.



In [ ]:
import librosa
import matplotlib.pyplot as plt
from IPython.display import Audio, display

# Ensure SENTENCE, processor, model, and voice_preset are defined from previous cells
# If not, they would need to be re-initialized here.
# SENTENCE is defined as "Welcome to KAUST Academy." in a previous cell.
# processor, model, and voice_preset are defined in cell 54azVlImYDMQ.

# process and move inputs to CUDA
inputs = processor(SENTENCE, voice_preset=voice_preset)
inputs = {k: v.to("cuda") for k, v in inputs.items()}

# generate audio
bark_audio_array_comparison = model.generate(**inputs)

# move output to CPU for further processing
bark_audio_array_comparison = bark_audio_array_comparison.cpu().numpy().squeeze()
bark_sr_comparison = model.generation_config.sample_rate

# Visualize the utterance in a spectrogram
fig, ax = plt.subplots(figsize=(10, 4))
generate_mel_spectrogram(bark_audio_array_comparison, sr=bark_sr_comparison, ax=ax)
ax.set_title(f'Bark Generated Mel Spectrogram: "{SENTENCE}"')
plt.tight_layout()
plt.show()

display(Audio(data=bark_audio_array_comparison, rate=bark_sr_comparison))
print("Generated Bark audio for comparison with MMS-TTS.")

#### Generating a voice clone sample from your input sample

In [ ]:
from scipy.io.wavfile import write as write_wav
from IPython.display import Audio, display

# Text to be synthesized
text_to_synthesize = "It took me quite a long time to develop a voice, and now that I have it I'm not going to be silent."

# Ensure processor and model are defined from previous cells (e.g., 54azVlImYDMQ)
# If they were not, you would need to uncomment and run the following lines:
# from transformers import AutoProcessor, BarkModel
# processor = AutoProcessor.from_pretrained("suno/bark")
# model = BarkModel.from_pretrained("suno/bark").to("cuda")

# Generate audio using the recorded voice as history_prompt with the transformers API
# 'train_audio.wav' was created in previous cells from your recording
inputs = processor(text_to_synthesize, history_prompt="train_audio.wav")
inputs = {k: v.to("cuda") for k, v in inputs.items()}

bark_audio_array = model.generate(**inputs)
bark_audio_array = bark_audio_array.cpu().numpy().squeeze()
bark_sr = model.generation_config.sample_rate

# Save the generated audio to a WAV file
output_filename = "output-en.wav"
write_wav(output_filename, bark_sr, bark_audio_array)

print(f"Generated voice clone saved to {output_filename}")

# Display the audio in the notebook
display(Audio(output_filename, rate=bark_sr))

#### Task: Finding markers of non-human speech in visualizations

One way we might reveal that an utterance is from a TTS system rather than real human voice is via visualizing the time-domain and spectrogram plots to find indicators of TTS.

Create an utterance from your voice clone that fairly closely matches the words in the real voice audio sample you provided.

Visualize both the time domain and spectrogram of your original sample and the voice clone reproduction of your sample.

Adjust your plots to show any indicators the voice is not human (e.g. by zooming in or focusing on particular time/frequency ranges)

Describe and annotate in your plots as best you can any indicators you can find that would support a claim that the voice clone utterance does not come from a human.

In [ ]:
import librosa
import matplotlib.pyplot as plt
from IPython.display import Audio, display

# Load original recorded audio
original_file_path = "train_audio.wav"
original_wav, original_sr = librosa.load(original_file_path, sr=None)

# Load voice clone audio (generated in the previous cell 'a756eb95')
clone_file_path = "output-en.wav"
clone_wav, clone_sr = librosa.load(clone_file_path, sr=None)

# Create a figure with 4 subplots (2 for original, 2 for clone)
fig, axs = plt.subplots(4, 1, figsize=(12, 16))

# Plot original waveform
visualize_waveform(original_wav, sr=original_sr, ax=axs[0])
axs[0].set_title('Original Recorded Audio - Waveform')

# Plot original mel spectrogram
generate_mel_spectrogram(original_wav, sr=original_sr, ax=axs[1])
axs[1].set_title('Original Recorded Audio - Mel Spectrogram')

# Plot voice clone waveform
visualize_waveform(clone_wav, sr=clone_sr, ax=axs[2])
axs[2].set_title('Voice Clone Audio - Waveform')

# Plot voice clone mel spectrogram
generate_mel_spectrogram(clone_wav, sr=clone_sr, ax=axs[3])
axs[3].set_title('Voice Clone Audio - Mel Spectrogram')

plt.tight_layout()
plt.show()

# Provide observations on potential TTS indicators
print("\n--- Observations on TTS Indicators ---")
print("When comparing the original recorded audio to the voice clone, several indicators might suggest the latter is machine-generated:")
print("1.  **Waveform Uniformity:** The voice clone's waveform might appear more regular or 'perfect' with less natural variation in amplitude compared to the human recording, which often shows subtle irregularities.")
print("2.  **Spectral Smoothness/Artifacts:** The mel spectrogram of the voice clone might exhibit smoother, sometimes unnaturally uniform transitions between phonemes or occasional 'banding' or 'blurring' that is less common in natural speech.")
print("3.  **Lack of Fine Details:** Human speech often contains subtle acoustic details like breath sounds, lip smacks, or slight vocal fry that might be absent or oversimplified in the cloned audio's waveform and spectrogram.")
print("4.  **Intonation Patterns:** While Bark is good at prosody, the intonation in the clone might still sound slightly less natural or more predictable, which can sometimes be seen in the fundamental frequency (F0) contour (though F0 is not explicitly plotted here, its impact can be inferred from spectral energy distribution).")
print("5.  **Steady-State Regions:** Sustained vowels or fricatives in TTS might show more stable and less dynamic spectral patterns than human speech, which tends to have more micro-variations.")
print("6.  **Silence Gaps:** The silence regions in synthesized speech might be 'too perfect' or abruptly cut off, lacking the natural reverb or background noise present in a real recording.")
print("--------------------------------------")

#### Task: Generate and visualize an Arablc voice clone sample


Now, generate audio in the Arabic language using your cloned voice. For the `text` argument, you may choose what utterance to render, but ensure the text matches the input language you are using.

In [ ]:
from scipy.io.wavfile import write as write_wav
from IPython.display import Audio, display

# Arabic text to be synthesized
ARABIC_TEXT_TO_SYNTHESIZE = "أهلاً بكم في أكاديمية الملك عبد الله للعلوم والتقنية."

# Ensure processor and model are defined from previous cells (e.g., 54azVlImYDMQ)
# 'train_audio.wav' was created in previous cells from your recording
inputs = processor(ARABIC_TEXT_TO_SYNTHESIZE, history_prompt="train_audio.wav")
inputs = {k: v.to("cuda") for k, v in inputs.items()}

bark_arabic_audio_array = model.generate(**inputs)
bark_arabic_audio_array = bark_arabic_audio_array.cpu().numpy().squeeze()
bark_arabic_sr = model.generation_config.sample_rate

# Save the generated Arabic audio to a WAV file
output_arabic_filename = "output-ara-clone.wav"
write_wav(output_arabic_filename, bark_arabic_sr, bark_arabic_audio_array)

print(f"Generated Arabic voice clone saved to {output_arabic_filename}")

# Display the audio in the notebook
display(Audio(output_arabic_filename, rate=bark_arabic_sr))

Plot and compare your English and Arabic utterances. Do you notice any differences in the low-level acoustic features of your cloned voice when speaking other languages? Does the naturalness or expressivity of your voice seem affected?

In [ ]:
import librosa
import matplotlib.pyplot as plt

# Load English voice clone audio
english_clone_file = "output-en.wav"
english_clone_wav, english_clone_sr = librosa.load(english_clone_file, sr=None)

# Load Arabic voice clone audio
arabic_clone_file = "output-ara-clone.wav"
arabic_clone_wav, arabic_clone_sr = librosa.load(arabic_clone_file, sr=None)

# Create a figure with two subplots for comparison
fig, axs = plt.subplots(2, 1, figsize=(12, 10))

# Plot English voice clone mel spectrogram
generate_mel_spectrogram(english_clone_wav, sr=english_clone_sr, ax=axs[0])
axs[0].set_title('English Voice Clone - Mel Spectrogram')

# Plot Arabic voice clone mel spectrogram
generate_mel_spectrogram(arabic_clone_wav, sr=arabic_clone_sr, ax=axs[1])
axs[1].set_title('Arabic Voice Clone - Mel Spectrogram')

plt.tight_layout()
plt.show()

print("\n--- Comparison of English and Arabic Voice Clones ---")
print("**Low-level Acoustic Features:**")
print("1.  **Formant Structures:** We can observe distinct formant patterns (dark horizontal bands) in both languages. However, the exact frequencies and transitions will differ due to the unique phonetic inventories of English and Arabic.")
print("2.  **Energy Distribution:** The distribution of energy across frequency bands might show differences. Arabic, with its pharyngeal and emphatic consonants, might exhibit more energy in lower frequency bands or distinct spectral characteristics for these sounds compared to English.")
print("3.  **Rhythm and Prosody:** While not directly visible in the spectrogram without F0 analysis, the temporal patterns (duration of sounds and pauses) might reflect differences in the natural rhythm and intonation contours of the two languages.")
print("\n**Naturalness or Expressivity:**")
print("1.  **Prosodic Transfer:** The Bark model is designed to transfer prosody. Listening reveals if the model successfully adapted the prosodic style from the English source audio (train_audio.wav) to the Arabic utterance. If the source audio had a particular intonation, it might be partially transferred to the Arabic, sometimes leading to an accent or unnaturalness if not perfectly adapted.")
print("2.  **Pronunciation Quality:** The naturalness can be heavily affected by how well the model pronounces specific phonemes in Arabic, especially those not present in English (e.g., guttural sounds). Mistakes in pronunciation can reduce naturalness.")
print("3.  **Expressiveness:** If the source audio was expressive, does the Arabic clone retain that expressiveness? Often, cross-lingual voice cloning can struggle to maintain the same level of natural expressivity, potentially resulting in a more monotone or less dynamic delivery in the target language.")
print("----------------------------------------------------")

### Task: Generate audio that a normal TTS can't produce


Bark can do much more than a standard TTS system. Think of an application where you might want a mix of speech and other sounds, or heavily modified prosody. Generate at least one sample with Bark that a normal TTS system would not produce, and describe how Bark might support the application / use case you picked.

In [ ]:
from scipy.io.wavfile import write as write_wav
from IPython.display import Audio, display

# Text with a special token for laughter, which Bark can interpret
# Bark can also interpret other non-speech sounds like [sigh], [crying], etc.
EXPRESSIVE_TEXT = "I finally understood the joke! [laughs] That's hilarious!"

# Ensure processor and model are defined from previous cells (e.g., 54azVlImYDMQ)
# Use a standard voice preset for this demonstration

inputs_expressive = processor(EXPRESSIVE_TEXT, voice_preset=voice_preset)
inputs_expressive = {k: v.to("cuda") for k, v in inputs_expressive.items()}

bark_expressive_audio_array = model.generate(**inputs_expressive)
bark_expressive_audio_array = bark_expressive_audio_array.cpu().numpy().squeeze()
bark_expressive_sr = model.generation_config.sample_rate

# Save the generated audio to a WAV file
output_expressive_filename = "output_bark_expressive.wav"
write_wav(output_expressive_filename, bark_expressive_sr, bark_expressive_audio_array)

print(f"Generated expressive Bark audio saved to {output_expressive_filename}")

# Display the audio in the notebook
display(Audio(output_expressive_filename, rate=bark_expressive_sr))

print("\n--- Application / Use Case ---")
print("This sample demonstrates Bark's ability to synthesize non-speech sounds like laughter, which is typically beyond the scope of traditional TTS systems that focus solely on speech. This capability is highly valuable for applications requiring more emotive and human-like interactions:")
print("1.  **Audiobooks and Podcasts:** Adding realistic non-speech cues (like laughter, sighs, or gasps) can significantly enhance the immersion and expressiveness of narrated content, making it more engaging for listeners.")
print("2.  **Virtual Assistants and Chatbots:** Implementing emotionally resonant responses or acknowledging user input with natural non-speech sounds can make virtual assistants feel more empathetic and less robotic.")
print("3.  **Gaming and Entertainment:** Characters could have more nuanced vocalizations, adding depth to their personalities without requiring extensive voice acting for every single sound effect.")
print("4.  **Accessibility Tools:** For users with visual impairments, a TTS system that can describe non-speech auditory events within text can provide a richer and more complete understanding of content.")
print("Traditional TTS systems would require separate audio files for such sounds, making the integration complex and less fluid. Bark's end-to-end generation of both speech and non-speech elements from text simplifies this process and leads to a more cohesive output.")